# Pipeline DEF-rgbtcc (Small Images) - Notebook 01: Pré-Transformação e Padronização de Recortes

Este notebook implementa a ingestão, equalização radiométrica local (CLAHE) e padronização geométrica de **recortes de imagem de pequena dimensão com poucas pessoas**.

> **Diretriz Didática:** Cada etapa contém uma explicação simples e direta sobre a escolha da lógica do código, seu fundamento técnico e o efeito prático esperado no resultado final.

## 1. Setup do Ambiente e Configuração de Diretórios

**O que este código faz:**
Importa as bibliotecas fundamentais de visão computacional (`OpenCV`, `NumPy`, `Matplotlib`), define o diretório de trabalho e cria a pasta de saída para o contrato de dados (`output/01_pre_transformacao/`).

**Por que esta lógica foi escolhida?**
O isolamento estrito de caminhos garante que os arquivos gerados em recortes pequenos fiquem separados das execuções de alta resolução (8000x6000 px). Além disso, definir `MPLCONFIGDIR` previne avisos de permissão ao salvar figuras no ambiente virtual.

**Efeito prático no resultado:**
O ambiente fica preparado e os diretórios de saída são criados automaticamente caso ainda não existam.

In [ ]:
import os
import sys
import json
from pathlib import Path

# Configurar diretório de cache do Matplotlib
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

import cv2
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT_DIR = NOTEBOOK_DIR.parent.parent
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output" / "01_pre_transformacao"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELO_NOME = "DEF-rgbtcc-small-images"

print("=" * 65)
print(f"[*] Pipeline Ativo:         {MODELO_NOME}")
print(f"[*] Diretório de Trabalho:  {NOTEBOOK_DIR}")
print(f"[*] Diretório de Insumos:   {INPUT_DIR}")
print(f"[*] Diretório de Saída:     {OUTPUT_DIR}")
print("=" * 65)

## 2. Seleção e Carregamento da Amostra Multimodal

**O que este código faz:**
Permite escolher interativamente entre 4 recortes reais pré-curados com diferentes quantidades de pessoas:
1. `calcada_9_pessoas`: Calçada de Pedestres no Solo (**9 pessoas** - cenário padrão).
2. `lateral_5_pessoas`: Lateral Direita com Luminárias (**5 pessoas**).
3. `area_vazia_0_pessoas`: Área Vazia - Telhado/Céu (**0 pessoas** - controle negativo).
4. `canto_19_pessoas`: Canto da Rua Inferior Direito (**19 pessoas**).

**Por que esta lógica foi escolhida?**
Em vez de forçar o usuário a abrir um editor externo para cortar a imagem toda vez que quiser testar uma cena diferente, disponibilizamos amostras curadas prontas com coordenadas físicas e anotações reais de cabeças (*Ground Truth*).

**Efeito prático no resultado:**
Carrega os pares óptico e térmico correspondentes e exibe os metadados da amostra selecionada.

In [ ]:
# Escolha da amostra para processamento:
# Opções disponíveis: 'calcada_9_pessoas', 'lateral_5_pessoas', 'area_vazia_0_pessoas', 'canto_19_pessoas'
AMODELO_SELECIONADA = 'calcada_9_pessoas'

samples_dir = INPUT_DIR / "samples"
p_rgb = samples_dir / f"{AMODELO_SELECIONADA}_rgb.jpg"
p_th = samples_dir / f"{AMODELO_SELECIONADA}_thermal.jpg"
p_gt = samples_dir / f"{AMODELO_SELECIONADA}_gt.json"

assert p_rgb.exists(), f"Erro: Insumo RGB não encontrado em {p_rgb}"
assert p_th.exists(), f"Erro: Insumo Térmico não encontrado em {p_th}"

# Carregar imagens em BGR e converter para RGB
img_rgb_raw = cv2.cvtColor(cv2.imread(str(p_rgb)), cv2.COLOR_BGR2RGB)
img_th_raw = cv2.cvtColor(cv2.imread(str(p_th)), cv2.COLOR_BGR2RGB)

with open(p_gt, "r", encoding="utf-8") as f:
    gt_meta = json.load(f)

h_raw, w_raw = img_rgb_raw.shape[:2]
real_count = gt_meta.get("total_pessoas_real", 0)

print("=" * 65)
print(f"[✓] Amostra Ativa: '{gt_meta.get('nome')}'")
print(f"    ├─ Resolução Original do Recorte: {w_raw}x{h_raw} px")
print(f"    ├─ Total de Pessoas Reais (GT):   {real_count} pessoas")
print(f"    └─ Coordenadas na Cena Global:    {gt_meta.get('coordenadas_roi_global')}")
print("=" * 65)

## 3. Visualização do Par Bruto e Ground Truth de Referência

**O que este código faz:**
Desenha círculos verdes sobre as cabeças de pedestres anotadas no Ground Truth humano e plota lado a lado a imagem óptica anotada e a imagem térmica bruta.

**Por que esta lógica foi escolhida?**
A inspeção visual inicial é o primeiro pilar de qualidade em visão computacional. Ela permite auditar se os pontos anotados realmente coincidem com pedestres visíveis antes de qualquer processamento matemático.

**Efeito prático no resultado:**
Dois painéis lado a lado mostrando o recorte RGB com os pedestres marcados e a cena correspondente no espectro infravermelho.

In [ ]:
vis_gt = img_rgb_raw.copy()
for pt in gt_meta.get("pontos_relativos", []):
    cv2.circle(vis_gt, (pt["x"], pt["y"]), 5, (0, 255, 0), -1)
    cv2.circle(vis_gt, (pt["x"], pt["y"]), 6, (0, 0, 255), 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(vis_gt)
axes[0].set_title(f"A. Recorte Óptico RGB (Ground Truth: {real_count} pessoas)", fontsize=11, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(img_th_raw)
axes[1].set_title(f"B. Recorte Térmico LWIR Bruto ({w_raw}x{h_raw} px)", fontsize=11, fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 4. Equalização Térmica Local Adaptativa (CLAHE)

**O que este código faz:**
Aplica o algoritmo CLAHE (*Contrast Limited Adaptive Histogram Equalization*) no canal de luminância da imagem térmica, utilizando um gradeado local `tileGridSize=(4, 4)` e limite de contraste `clipLimit=2.0`.

**Por que esta lógica foi escolhida?**
Em imagens térmicas globais grandes (640x512 ou 8000x6000), grades de 8x8 são comuns. No entanto, em **recortes pequenos (~300 px)**, blocos grandes provocam perda de detalhes finos, enquanto um limite de contraste muito alto amplifica o ruído granulado do sensor. O grid 4x4 com clipLimit moderado (2.0) realça com máxima precisão o calor emitido pelo corpo humano sem degradar o fundo frio do asfalto.

**Efeito prático no resultado:**
As silhuetas térmicas dos pedestres tornam-se nítidas e com alto contraste, facilitando a extração de características pela rede neural.

In [ ]:
# Conversão para o espaço LAB para equalizar estritamente a luminosidade
lab_th = cv2.cvtColor(img_th_raw, cv2.COLOR_RGB2LAB)
l_channel, a_channel, b_channel = cv2.split(lab_th)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
l_eq = clahe.apply(l_channel)

lab_eq = cv2.merge((l_eq, a_channel, b_channel))
img_th_clahe = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_th_raw)
axes[0].set_title("A. Térmica Bruta (Baixo Contraste)", fontsize=11, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(img_th_clahe)
axes[1].set_title("B. Térmica com CLAHE Local (Silhuetas Destacadas)", fontsize=11, fontweight="bold", color="darkgreen")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Padronização Dimensional para Redes Neurais (Múltiplos de 32 Pixels)

**O que este código faz:**
Calcula as dimensões divisíveis por 32 mais próximas da largura e altura originais do recorte e ajusta as imagens via interpolação bicúbica de alta fidelidade:
$$\text{dim}_{32} = \left\lceil \frac{\text{dim}}{32} \right\rceil \times 32$$

**Por que esta lógica foi escolhida?**
Redes Convolucionais profundas (backbones VGG-19, ResNet) e Vision Transformers realizam divisões sucessivas por 2 ao longo dos blocos de processamento (geralmente 5 estágios de redução: $2^5 = 32$). Se a imagem de entrada tiver dimensão ímpar ou não divisível por 32, as operações de *pooling* e interpolação de retorno quebram as dimensões dos tensores ou introduzem artefatos assimétricos de borda.

**Efeito prático no resultado:**
As imagens mantêm sua proporção quase intacta, mas agora possuem dimensões matematicamente compatíveis com qualquer backbone de Deep Learning.

In [ ]:
# Cálculo das dimensões ótimas múltiplas de 32
w_32 = int(((w_raw + 31) // 32) * 32)
h_32 = int(((h_raw + 31) // 32) * 32)

img_rgb_pad = cv2.resize(img_rgb_raw, (w_32, h_32), interpolation=cv2.INTER_CUBIC)
img_th_pad = cv2.resize(img_th_clahe, (w_32, h_32), interpolation=cv2.INTER_CUBIC)

print("=" * 65)
print(f"[*] Resolução Original do Recorte:   {w_raw}x{h_raw} px")
print(f"[✓] Resolução Ajustada (Divisível por 32): {w_32}x{h_32} px")
print(f"    ├─ Fator de escala horizontal: {w_32 / w_raw:.4f}")
print(f"    └─ Fator de escala vertical:   {h_32 / h_raw:.4f}")
print("=" * 65)

## 6. Auditoria de Co-registro e Blend 50/50

**O que este código faz:**
Gera uma imagem de auditoria por sobreposição de transparência equilibrada:
$$\text{Blend} = 0.50 \times \text{RGB} + 0.50 \times \text{Térmica}$$

**Por que esta lógica foi escolhida?**
Em tarefas multimodais (RGBT), o alinhamento espacial entre o canal óptico e o térmico precisa ser sub-pixel. Se a silhueta térmica estiver deslocada em relação à imagem visual, a rede neural aprenderá uma representação conflitante. O blend permite comprovar visualmente que os corpos no infravermelho coincidem perfeitamente com os corpos na foto.

**Efeito prático no resultado:**
Exibição do blend onde as silhuetas quentes vestem perfeitamente os pedestres do espectro visual.

In [ ]:
blend_audit = cv2.addWeighted(img_rgb_pad, 0.50, img_th_pad, 0.50, 0)

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(blend_audit)
ax.set_title(f"Auditoria de Alinhamento Multimodal (50% RGB / 50% Térmica) - {w_32}x{h_32} px", fontsize=12, fontweight="bold")
ax.axis("off")

plt.tight_layout()
blend_path = OUTPUT_DIR / "blend_auditoria.jpg"
plt.savefig(str(blend_path), dpi=150, bbox_inches="tight")
plt.show()

print(f"[✓] Painel de blend salvo em: {blend_path}")

## 7. Exportação do Contrato de Insumos Padronizados e Metadados

**O que este código faz:**
Grava os arquivos finais em `output/01_pre_transformacao/` no formato padronizado que será consumido de forma autônoma pelo **Notebook 02**:
1. `rgb_preprocessed.jpg`: Recorte visual padronizado.
2. `thermal_preprocessed.jpg`: Recorte térmico com CLAHE local e dimensões compatíveis.
3. `metadata_preprocessing.json`: Metadados completos da transformação e do Ground Truth.

**Por que esta lógica foi escolhida?**
O padrão arquitetural de **Data Contracts** garante isolamento completo entre as etapas. O Estágio 2 não precisa saber como a imagem foi equalizada ou de onde veio; ele apenas consome um par de imagens com garantia de dimensões e alinhamento válidos.

**Efeito prático no resultado:**
Arquivos gravados com sucesso e prontos para alimentar as redes neurais.

In [ ]:
# Salvar imagens do contrato
p_out_rgb = OUTPUT_DIR / "rgb_preprocessed.jpg"
p_out_th = OUTPUT_DIR / "thermal_preprocessed.jpg"

cv2.imwrite(str(p_out_rgb), cv2.cvtColor(img_rgb_pad, cv2.COLOR_RGB2BGR))
cv2.imwrite(str(p_out_th), cv2.cvtColor(img_th_pad, cv2.COLOR_RGB2BGR))

# Mapear pontos de ground truth para o espaço padronizado w_32 x h_32
scale_x = w_32 / w_raw
scale_y = h_32 / h_raw
pts_scaled = [
    {"id": pt["id"], "x": int(round(pt["x"] * scale_x)), "y": int(round(pt["y"] * scale_y))}
    for pt in gt_meta.get("pontos_relativos", [])
]

metadata = {
    "modelo_notebook": MODELO_NOME,
    "amostra_ativa": AMODELO_SELECIONADA,
    "nome_amostra": gt_meta.get("nome"),
    "resolucao_original": [w_raw, h_raw],
    "resolucao_padronizada": [w_32, h_32],
    "fator_escala": [scale_x, scale_y],
    "pessoas_reais_ground_truth": real_count,
    "pontos_ground_truth": pts_scaled,
    "arquivos_gerados": {
        "rgb": str(p_out_rgb.relative_to(NOTEBOOK_DIR)),
        "thermal": str(p_out_th.relative_to(NOTEBOOK_DIR)),
        "blend": str(blend_path.relative_to(NOTEBOOK_DIR))
    }
}

metadata_path = OUTPUT_DIR / "metadata_preprocessing.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("=" * 65)
print("     ESTÁGIO 1 CONCLUÍDO: CONTRATO DE DADOS GERADO COM SUCESSO")
print("=" * 65)
print(f"1. RGB Padronizado:     {p_out_rgb}")
print(f"2. Térmica Padronizada: {p_out_th}")
print(f"3. Blend de Auditoria:  {blend_path}")
print(f"4. Metadados do Insumo: {metadata_path}")
print("=" * 65)